# Week 5 Assignment: Building Your First Supervised Machine Learning Model

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

## Objective
This notebook builds and compares two supervised **regression** models using the ND-GAIN Country Index dataset. The target is a country's climate-vulnerability score. Candidate predictors are selected from the available numeric columns while excluding the target and closely related vulnerability-component variables where possible.

The notebook:
1. Loads and inspects the dataset.
2. Identifies the target and input features.
3. Splits observations into training and test sets.
4. Trains Linear Regression and Random Forest Regression.
5. Evaluates predictions using MAE, RMSE, and R².
6. Compares model performance and discusses limitations.

**Important:** The ND-GAIN dataset's column names and downloadable files can change between releases. The loading and column-selection code checks the available columns and prints its choices. Review the selected target and features before interpreting results.

## How to run this notebook
1. Open the notebook in Google Colab or Jupyter Notebook.
2. Run the cells from top to bottom in order.
3. If automatic dataset download fails, upload the relevant CSV/ZIP file as described in the data-loading section.
4. Review the generated metric tables and plots; the final comparison is calculated from the actual test-set results.
5. Save the notebook after running it, then upload the `.ipynb` file to your course-project GitHub repository.


## 1. Import libraries

In [ ]:
import os
import glob
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
RANDOM_STATE = 42

## 2. Load the ND-GAIN dataset
The cell tries to download the ND-GAIN Country Index archive. If that fails, upload the downloaded ZIP or CSV file to the notebook session and run the cell again.

In [ ]:
DATA_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
ZIP_PATH = "ndgain_countryindex_2026.zip"
EXTRACT_DIR = "ndgain_data"

if not os.path.isdir(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)

if not glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True):
    try:
        if not os.path.exists(ZIP_PATH):
            urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("Downloaded and extracted the ND-GAIN archive.")
    except Exception as exc:
        print("Automatic download did not work:", exc)
        print("Please upload the ND-GAIN ZIP/CSV file and set its path below.")

csv_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
csv_files += [p for p in glob.glob("*.csv") if p not in csv_files]

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found. Upload the ND-GAIN CSV or ZIP file, "
        "then rerun this section."
    )

tables = {}
for path in csv_files:
    try:
        tables[path] = pd.read_csv(path)
        print(f"{path}: {tables[path].shape[0]} rows x {tables[path].shape[1]} columns")
    except Exception as exc:
        print(f"Could not read {path}: {exc}")

if not tables:
    raise ValueError("CSV files were found, but none could be read.")

# Prefer a table whose filename or columns mention vulnerability/readiness.
def table_score(item):
    path, frame = item
    text = (os.path.basename(path) + " " + " ".join(map(str, frame.columns))).lower()
    return int("vulnerab" in text) * 3 + int("readiness" in text) * 2 + int("country" in text)

selected_path, df = max(tables.items(), key=table_score)
print("\nSelected file:", selected_path)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

## 3. Identify the target and predictor features
The target is the climate vulnerability score. Column matching is case-insensitive and tolerates common naming variations. Predictor selection avoids country identifiers, year, the target itself, and variables explicitly named as vulnerability components to reduce direct target leakage.

In [ ]:
def normalized_name(name):
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())

column_lookup = {normalized_name(c): c for c in df.columns}

def find_column(candidates):
    # Exact normalized match first
    for candidate in candidates:
        key = normalized_name(candidate)
        if key in column_lookup:
            return column_lookup[key]
    # Then partial match
    for candidate in candidates:
        key = normalized_name(candidate)
        for normalized, original in column_lookup.items():
            if key and key in normalized:
                return original
    return None

target_col = find_column([
    "Vulnerability", "Vulnerability Score", "Vulnerability Index",
    "Climate Vulnerability", "Vulnerability score"
])

if target_col is None:
    possible = [c for c in df.columns if "vulnerab" in str(c).lower()]
    if possible:
        target_col = possible[0]

if target_col is None:
    raise ValueError(
        "Could not identify a vulnerability target column. Inspect df.columns "
        "and set target_col manually to the correct climate-vulnerability score."
    )

# Candidate predictors: readiness is commonly available and relevant to adaptation.
readiness_col = find_column(["Readiness", "Readiness Score", "Readiness Index"])
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Exclude likely identifiers, time, target, and direct vulnerability components.
excluded_terms = [
    "vulnerab", "exposure", "sensitivity", "adaptivecapacity",
    "countrycode", "iso3", "iso", "year", "rank"
]
feature_cols = []
if readiness_col is not None and readiness_col != target_col and readiness_col in numeric_cols:
    feature_cols.append(readiness_col)

for col in numeric_cols:
    norm = normalized_name(col)
    if col == target_col or col in feature_cols:
        continue
    if any(term in norm for term in excluded_terms):
        continue
    feature_cols.append(col)

# Avoid using a large number of arbitrary columns; prefer readiness plus a small set
# of general numeric indicators. If there are few candidates, use all eligible columns.
feature_cols = feature_cols[:10]

if not feature_cols:
    raise ValueError(
        "No suitable numeric predictors were found. Inspect numeric columns and "
        "manually set feature_cols to relevant indicators."
    )

print("Target/output:", target_col)
print("Predictor/features:", feature_cols)
print("Problem type: regression, because the target is a continuous numerical score.")

# Convert selected fields to numeric and remove rows without a known target.
model_data = df[[target_col] + feature_cols].copy()
for col in model_data.columns:
    model_data[col] = pd.to_numeric(model_data[col], errors="coerce")
model_data = model_data.replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=[target_col])

# Remove predictors that contain no usable values.
feature_cols = [c for c in feature_cols if model_data[c].notna().any()]
model_data = model_data[[target_col] + feature_cols]

if len(model_data) < 10:
    raise ValueError("Too few usable rows remain for a reliable train/test split.")

X = model_data[feature_cols]
y = model_data[target_col]

print(f"Usable rows: {len(model_data)}")
display(model_data.head())
display(model_data[[target_col] + feature_cols].describe().T)

## 4. Split into training and test sets
The test set is held back from model fitting so it can provide an estimate of performance on unseen observations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Train two supervised regression models
Linear Regression provides a simple baseline. Random Forest Regression can model nonlinear patterns and interactions. Missing feature values are imputed using the training data; Linear Regression also standardizes features.

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    ))
])

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 6. Predict on the test set

In [ ]:
predictions = {
    "Linear Regression": linear_model.predict(X_test),
    "Random Forest": forest_model.predict(X_test)
}

comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Linear Regression": predictions["Linear Regression"],
    "Random Forest": predictions["Random Forest"]
}, index=y_test.index)

display(comparison.head(10))

## 7. Evaluate model performance
Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) measure prediction error; lower values are better. R² measures how much variation in the target is explained by the model; values closer to 1 generally indicate better fit, while a negative test R² means the model performs worse than predicting the test-set mean.

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred)
    }

metrics = pd.DataFrame({
    name: evaluate(y_test, pred)
    for name, pred in predictions.items()
}).T

display(metrics.round(4))

## 8. Compare predictions visually

In [ ]:
plt.figure(figsize=(7, 6))
for name, pred in predictions.items():
    plt.scatter(y_test, pred, alpha=0.65, label=name)

low = min(float(y_test.min()), *(float(np.min(p)) for p in predictions.values()))
high = max(float(y_test.max()), *(float(np.max(p)) for p in predictions.values()))
plt.plot([low, high], [low, high], linestyle="--", label="Perfect prediction")
plt.xlabel("Actual climate vulnerability score")
plt.ylabel("Predicted climate vulnerability score")
plt.title("Actual vs predicted climate vulnerability")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Compare models and discuss results

In [ ]:
best_by_rmse = metrics["RMSE"].idxmin()
best_by_mae = metrics["MAE"].idxmin()
best_by_r2 = metrics["R²"].idxmax()

print(f"Lowest RMSE: {best_by_rmse}")
print(f"Lowest MAE: {best_by_mae}")
print(f"Highest test R²: {best_by_r2}")

print("\nInterpretation:")
print("- Lower MAE/RMSE means predictions are closer to the observed scores.")
print("- Higher R² indicates that the model explains more variation in the test data.")
print("- If the models disagree on the best model across metrics, report that trade-off.")
print("- These results show association/predictive performance, not causal effects.")

best_model = metrics["RMSE"].idxmin()
print(f"\nBased on RMSE, {best_model} performed best on this split.")
print("Use the metric table and plot above to support your final discussion.")

## 10. Limitations and responsible interpretation

- ND-GAIN scores are indicators, not complete descriptions of every country's climate risks.
- The result depends on the specific data release, chosen columns, missing-value handling, and random train/test split.
- A random split can place records from the same country in both sets if the dataset contains multiple years per country. For a time-series or country-generalization study, use a time-based or group-based split.
- Some indicators may be conceptually related to the target. Check the printed feature list and exclude any feature that directly contributes to the target score if strict leakage prevention is required.
- Predictive association does not establish causation and should not be used alone to allocate public resources or make policy decisions.

## Conclusion
This notebook completes a supervised learning workflow by defining a numerical target, selecting features, splitting data, training two regression models, predicting on held-out data, and comparing them with MAE, RMSE, and R². The stronger model is the one with lower prediction errors and a higher R², interpreted in light of the dataset and limitations.

## References
1. Notre Dame Global Adaptation Initiative (ND-GAIN), Country Index: https://gain.nd.edu/our-work/country-index/
2. Scikit-learn documentation, Linear Regression: https://scikit-learn.org/stable/modules/linear_model.html
3. Scikit-learn documentation, Random Forest Regression: https://scikit-learn.org/stable/modules/ensemble.html#random-forests
4. Scikit-learn documentation, Model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html


# Week 7: Exploring Pretrained Models

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

## 11. Selected pretrained model: ClimateBERT

**Model name:** `climatebert/distilroberta-base-climate-f`  
**Hugging Face model page:** https://huggingface.co/climatebert/distilroberta-base-climate-f

ClimateBERT is a pretrained language model based on DistilRoBERTa. It was further pretrained on climate-related text, including research-paper abstracts, corporate reports, company reports, and general news. Its model card identifies it as a fill-mask model. It predicts a suitable word or token to replace a `[MASK]` token in a sentence.

### Input and output

- **Input:** English text containing a mask token, for example: “Climate change increases [MASK] risks.”
- **Output:** Ranked candidate words/tokens that could fill the masked position, together with scores.
- **Model task:** Masked-language modelling (fill-mask), not direct numerical climate-risk prediction.

## 12. Relevance to climate governance and adaptation research

ClimateBERT could support the analysis of climate-policy documents, adaptation plans, government reports, environmental assessments, and climate-related news. For example, it could help researchers explore the language used when discussing vulnerability, mitigation, adaptation, emissions, and climate finance. It may be useful as an initial text-analysis component when comparing how institutions describe climate risks and adaptation priorities.

In the existing ND-GAIN project, the supervised machine-learning models use numerical country indicators to predict a climate-vulnerability score. ClimateBERT works with text instead. It could therefore complement the numerical model if climate-policy documents or reports were added as a separate source of information. It does not directly replace the regression models or calculate the ND-GAIN vulnerability score.

## 13. Difference from the models developed in Week 5

The Week 5 project used Linear Regression and Random Forest Regression. These models were trained on the selected numerical features and the ND-GAIN target variable using a training/test split. Their purpose was to predict a continuous climate-vulnerability score and evaluate the predictions using MAE, RMSE, and R².

ClimateBERT is different because it is already pretrained on large collections of climate-related text. It processes language and performs a fill-mask task. It does not learn the ND-GAIN numerical relationship from the project dataset, and it does not automatically output a country's vulnerability score.

## 14. Could pretrained models, generative AI, or NLP complement the project?

Yes. NLP models such as ClimateBERT could complement the project by extracting or exploring information from climate-policy documents and reports. Generative AI could assist with summarising long policy documents or organising themes for further human review. The extracted text-based information could later be compared with numerical indicators, provided that the text variables are carefully defined and validated.

These tools should be used as supporting methods. The numerical regression analysis remains necessary for the original prediction task.

## 15. Limitations and precautions

1. **Task limitation:** The selected checkpoint is a fill-mask model, not a ready-made climate-vulnerability scoring model.
2. **Language limitation:** Its usefulness may be reduced for documents written in languages or local contexts not adequately represented in its training data.
3. **Context and bias:** Predictions reflect patterns in training text and may reproduce omissions or biases in those sources.
4. **Interpretation:** A plausible predicted word is not evidence that a climate claim is scientifically correct.
5. **Local relevance:** Kenyan and African policy documents should be checked to ensure that the model represents local terminology and priorities.
6. **Validation:** Any extracted information should be checked against original documents and expert knowledge.
7. **Privacy and governance:** Sensitive or unpublished policy documents should not be uploaded to external services without permission.

## 16. Optional demonstration: fill-mask prediction

The following demonstration uses a short illustrative sentence. It does not train or fine-tune ClimateBERT and does not make a scientific claim.



In [ ]:
# Install the required package if it is not already installed.
!pip -q install transformers torch

from transformers import pipeline

climate_fill_mask = pipeline(
    "fill-mask",
    model="climatebert/distilroberta-base-climate-f"
)

example_text = "Climate change increases [MASK] risks."
demo_results = climate_fill_mask(example_text, top_k=5)

for result in demo_results:
    print(result["token_str"].strip(), round(result["score"], 4))



### Demonstration interpretation

The model returns candidate words for the masked position and a score for each candidate. The scores indicate the model's relative preference among the candidate completions; they are not probabilities that a climate statement is true. The example is only a demonstration of the model's text-completion function.

## 17. Conclusion

ClimateBERT is relevant to climate governance because it has been pretrained on climate-related text and can support exploratory analysis of policy documents, reports, and climate communication. Its fill-mask task differs from the Week 5 regression models, which use numerical country indicators to predict a continuous vulnerability score. The two approaches could complement one another if text-based evidence is added carefully, but ClimateBERT should not be treated as a substitute for validated numerical modelling or expert climate assessment.

## References

1. ClimateBERT model card, Hugging Face: https://huggingface.co/climatebert/distilroberta-base-climate-f
2. Webersinke, N., Kraus, M., Bingler, J. & Leippold, M. (2022). *ClimateBERT: A Pretrained Language Model for Climate-Related Text*. https://doi.org/10.48550/arXiv.2212.13631
3. ND-GAIN Country Index: https://gain.nd.edu/our-work/country-index/


# Week 9: Model Evaluation, Explainability and Deployment Readiness

**Project:** Using Machine Learning to Support Climate Governance and Adaptation Decision-Making  
**Student:** Erick Munala Sifuna

This section continues the Week 5 regression project. It evaluates the existing Linear Regression and Random Forest Regression pipelines on both training and held-out test data, examines generalisation, uses SHAP to explain the Random Forest, and saves/reloads the selected pipeline for a prediction on a new input row.

**Important:** Numerical results are generated when the notebook is run with the dataset. No performance values are invented in this prepared notebook.

## 18. Week 9.1 — Evaluation on training and test data

The appropriate metrics for this project are shown in the code cell below. The held-out test set is used to estimate performance on observations not used to fit the model.


In [ ]:
# Week 9.1 — Compare training and test performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

regression_models = {"Linear Regression": linear_model, "Random Forest": forest_model}
rows = []
for model_name, fitted_model in regression_models.items():
    for split_name, X_part, y_part in [
        ("Training", X_train, y_train),
        ("Test", X_test, y_test)
    ]:
        y_hat = fitted_model.predict(X_part)
        rows.append({
            "Model": model_name,
            "Data": split_name,
            "MAE": mean_absolute_error(y_part, y_hat),
            "RMSE": np.sqrt(mean_squared_error(y_part, y_hat)),
            "R2": r2_score(y_part, y_hat)
        })

train_test_metrics = pd.DataFrame(rows)
display(train_test_metrics.round(4))

# Select the model with the lowest held-out test RMSE.
test_rows = train_test_metrics[train_test_metrics["Data"] == "Test"]
best_model_name = test_rows.loc[test_rows["RMSE"].idxmin(), "Model"]
best_model = regression_models[best_model_name]
print("Selected model (lowest test RMSE):", best_model_name)


## 19. Week 9.2 — Overfitting and underfitting

A large performance gap between training and test data can indicate overfitting. Weak performance on both can indicate underfitting or limited predictive information. Interpret the computed values rather than assuming a result.


In [ ]:
# Week 9.2 — Diagnose possible overfitting or underfitting
for model_name in regression_models:
    part = train_test_metrics[train_test_metrics["Model"] == model_name].set_index("Data")
    train_rmse = part.loc["Training", "RMSE"]
    test_rmse = part.loc["Test", "RMSE"]
    train_r2 = part.loc["Training", "R2"]
    test_r2 = part.loc["Test", "R2"]
    gap = test_rmse - train_rmse
    print(f"\n{model_name}: training RMSE={train_rmse:.4f}, test RMSE={test_rmse:.4f}; "
          f"training R²={train_r2:.4f}, test R²={test_r2:.4f}")
    if gap > max(0.05 * abs(train_rmse), 1e-8):
        print("Interpretation: the test error is noticeably higher than training error; this is evidence of a possible generalisation gap/overfitting.")
    elif train_rmse > 0 and test_rmse <= 1.10 * train_rmse:
        print("Interpretation: training and test errors are relatively close; there is no strong overfitting signal from this split.")
    if train_r2 < 0.20 and test_r2 < 0.20:
        print("Both R² values are low; investigate possible underfitting, weak predictors, noisy targets, or a difficult target definition.")

print("These are diagnostic indicators, not proof. Confirm with cross-validation and, where relevant, country- or time-based validation.")


## 20. Week 9.3 — Explainability using SHAP

SHAP (SHapley Additive exPlanations) estimates how features contribute to model outputs. This section explains the Random Forest model. The SHAP ranking is an explanation of model behaviour, not a causal analysis. See the [SHAP TreeExplainer documentation](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html).


In [ ]:
# Week 9.3 — Explain predictions with SHAP (Random Forest)
# SHAP TreeExplainer explains tree ensembles. We explain the fitted forest inside the pipeline.
!pip -q install shap
import shap

rf_pipeline = forest_model
rf_imputer = rf_pipeline.named_steps["imputer"]
rf_core_model = rf_pipeline.named_steps["model"]
X_test_transformed = rf_imputer.transform(X_test)
explainer = shap.TreeExplainer(rf_core_model)
shap_values = explainer.shap_values(X_test_transformed)

# Regression produces one SHAP value per feature and observation.
if isinstance(shap_values, list):
    shap_array = np.asarray(shap_values[0])
else:
    shap_array = np.asarray(shap_values)
if shap_array.ndim == 3:
    shap_array = shap_array[:, :, 0]

mean_abs_shap = np.abs(shap_array).mean(axis=0)
shap_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Mean absolute SHAP value": mean_abs_shap
}).sort_values("Mean absolute SHAP value", ascending=False)
display(shap_importance)

plt.figure(figsize=(8, 5))
plot_data = shap_importance.sort_values("Mean absolute SHAP value")
plt.barh(plot_data["Feature"], plot_data["Mean absolute SHAP value"])
plt.xlabel("Mean absolute SHAP value (impact on model output scale)")
plt.title("Global feature importance — Random Forest")
plt.tight_layout()
plt.show()

print("Interpretation: larger mean absolute SHAP values indicate features that contribute more strongly, on average, to the model's predictions.")
print("SHAP importance describes model behaviour, not causation. Check whether leading indicators are substantively meaningful and whether any are proxies or leakage variables.")


### Interpreting the climate-governance model

The feature ranking produced above should be compared with climate-science and governance knowledge. Indicators connected to exposure, sensitivity, adaptive capacity, infrastructure, health, water, or institutional readiness may plausibly influence a climate-vulnerability estimate, depending on the exact columns in the downloaded ND-GAIN release. The actual ranking must be read from the notebook output; it should not be assumed in advance.

A high SHAP value means the model relies strongly on that feature for its predictions. It does **not** show that the feature causes vulnerability. Correlated indicators can share or redistribute importance, and a feature can appear important because of data construction or leakage. The project should therefore review the exact predictor definitions and consider a country-grouped or time-based validation strategy if multiple years per country are present.


## 21. Week 9.4 — Save, reload and predict on new data

The fitted pipeline is saved using Joblib, reloaded, and applied to one held-out input row as a demonstration. In a real deployment, replace this demonstration row with a genuinely new record using the same feature schema. See the [scikit-learn model persistence guidance](https://scikit-learn.org/stable/model_persistence.html).


In [ ]:
# Week 9.4 — Save, reload, and predict on a new input row
# The complete pipeline is saved so the same imputation/scaling steps are retained.
import joblib
from pathlib import Path

artifact_path = Path("climate_governance_best_model.joblib")
joblib.dump({
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": list(feature_cols),
    "target_name": target_col,
    "project": "ND-GAIN climate vulnerability regression"
}, artifact_path)
print("Saved model artifact:", artifact_path.resolve())

loaded_artifact = joblib.load(artifact_path)
loaded_model = loaded_artifact["model"]

# Demonstration input: one held-out row, treated as a new incoming record.
# Replace this row with a genuinely new country/observation when available.
new_observation = X_test.iloc[[0]].copy()
new_prediction = loaded_model.predict(new_observation)
print("Input observation:")
display(new_observation)
print("Predicted climate-vulnerability score:", float(new_prediction[0]))
print("Target scale/range should be checked against the dataset documentation before practical use.")


## 22. Deployment, monitoring and limitations

### Practical deployment and post-deployment monitoring

**Possible practical use:** the saved pipeline could be exposed through a secure dashboard or API used by climate-planning analysts. An authorised user would submit the same predictor fields and receive an estimated vulnerability score to support screening and prioritisation. The estimate should support—not replace—expert assessment and locally grounded climate-risk analysis.

**What to monitor after deployment:**
- Missing values, invalid ranges, changes in units, and changes in the distribution of input indicators (data drift).
- Prediction distributions and, when verified outcomes become available, MAE, RMSE and R² over time (performance drift).
- Performance across regions, income groups and countries to detect uneven errors or systematic bias.
- Changes in ND-GAIN definitions, data releases, collection methods and policy context.
- Whether analysts are using predictions appropriately and whether decisions are being reviewed by domain experts.
- Model version, data snapshot, dependencies, access controls, audit logs and reproducibility.

**Limitations and precautions:** this model predicts an index-derived score, not a complete local climate-risk assessment. Country-level indicators can hide sub-national differences; correlated predictors and data leakage can distort importance; a random split may overstate performance if records from the same country appear in both sets. Before deployment, validate on later years or held-out countries, document uncertainty, review fairness and obtain domain-expert approval. Load only trusted model files and keep compatible software versions.



## 23. Week 9 conclusion

This assignment adds a train/test generalisation check, model explainability, model persistence and a new-input prediction demonstration to the existing project. Final conclusions must be written after running the notebook and reviewing the actual metric table, overfitting diagnostics and SHAP feature ranking. Do not report uncomputed metrics or treat model explanations as causal evidence.
